In [20]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier


## Dataset Load

In [2]:
df = pd.read_csv("gym_members_exercise_tracking.csv")

## Feature Enginnering

In [3]:
def bmi_category(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

df['BMI_Category'] = df['BMI'].apply(bmi_category)

In [4]:
df['Heart_Rate_Reserve'] = df['Max_BPM'] - df['Resting_BPM']

In [5]:
df['HR_Intensity'] = df['Avg_BPM'] / df['Max_BPM']

In [6]:
def intensity_category(value):
    if value < 0.70:
        return 'Low'
    elif value < 0.85:
        return 'Moderate'
    else:
        return 'High'

df['Workout_Intensity'] = df['HR_Intensity'].apply(intensity_category)

In [7]:
df['Calories_Per_Hour'] = (df['Calories_Burned'] / df['Session_Duration (hours)']) 

In [8]:
df['Workout_Load'] = (df['Session_Duration (hours)'] * df['Avg_BPM'])

In [9]:
df['Weekly_Workout_Hours'] = (df['Workout_Frequency (days/week)']* df['Session_Duration (hours)'])

In [10]:
df['Risk_Score'] = 0

df.loc[df['BMI'] >= 30, 'Risk_Score'] += 1
df.loc[df['Fat_Percentage'] >= 30, 'Risk_Score'] += 1
df.loc[df['HR_Intensity'] >= 0.85, 'Risk_Score'] += 1
df.loc[df['Resting_BPM'] >= 70, 'Risk_Score'] += 1
df.loc[df['Workout_Load'] >= 200, 'Risk_Score'] += 1

In [11]:
def fitness_risk(score):

    if score <= 1:
        return 'Low'

    elif score == 2:
        return 'Moderate'

    else:
        return 'High'


df['Fitness_Risk'] = df['Risk_Score'].apply(fitness_risk)

In [12]:
df.head().T

,0,1,2,3,4
Age,56,46,32,25,38
Gender,Male,Female,Female,Male,Male
Weight (kg),88.3,74.9,68.1,53.2,46.1
Height (m),1.71,1.53,1.66,1.7,1.79
Max_BPM,180,179,167,190,188
Avg_BPM,157,151,122,164,158
Resting_BPM,60,66,54,56,68
Session_Duration (hours),1.69,1.3,1.11,0.59,0.64
Calories_Burned,1313.0,883.0,677.0,532.0,556.0
Workout_Type,Yoga,HIIT,Cardio,Strength,Strength


# Data Spliting

In [13]:
x = df.drop(['Fitness_Risk','Risk_Score', 'Calories_Burned','Calories_Per_Hour'], axis=1)

risk_mapping = {'Low': 0,'Moderate': 1,'High': 2}

y = df['Fitness_Risk'].map(risk_mapping)

## Performing Train Test Split

In [15]:
xtrain , xtest , ytrain , ytest = train_test_split(x,y,test_size = .25, random_state = 42)

## Encoding Apply


In [17]:
encoder = OneHotEncoder(sparse_output=False, drop='first')

categorical_columns = xtrain.select_dtypes(include='object').columns

for column in categorical_columns:

    train_encoded = encoder.fit_transform(xtrain[[column]])
    test_encoded = encoder.transform(xtest[[column]])

    train_df = pd.DataFrame(train_encoded,
                            columns=encoder.get_feature_names_out([column]),
                            index=xtrain.index)

    test_df = pd.DataFrame(test_encoded,
                           columns=encoder.get_feature_names_out([column]),
                           index=xtest.index)

    xtrain = pd.concat([xtrain, train_df], axis=1)
    xtest = pd.concat([xtest, test_df], axis=1)

    xtrain.drop(column, axis=1, inplace=True)
    xtest.drop(column, axis=1, inplace=True)